# scikit-FIBERS: Multi-Thresholding Demonstration Notebook

This notebook demonstrates the multi-thresholding capabilities of the FIBERS algorithm, including:

1. **Multi-thresholding**: Support for both 2-group (1 threshold) and 3-group (2 thresholds) bins
2. **Fitness function exploration**: Tests both multivariate log-rank tests and pairwise comparisons for 3-group bins
3. **Bin injection mechanism**: Allows testing of ground truth bins against FIBERS-discovered bins
4. **Graphing and Evaluation**: Performance tracking, visualizations, comparison tools

## General process:
1. Generate simulated survival data (2-group or 3-group)
2. Train FIBERS with multi-thresholding enabled
3. Create optimal bins with known thresholds
4. Inject optimal bins and compare performance
5. Visualize results with Kaplan-Meier curves and area analysis

## Current Challenge:
Both multivariate log-rank tests and pairwise averaging approaches have limitations in properly handling the bias toward 3-group bins when 2-group solutions are optimal. The core problem remains unsolved and requires further research into alternative fitness metrics.

***
## Imports:

In [ ]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from src.skfibers.fibers import FIBERS
from src.skfibers.experiments.survival_sim_simple import survival_data_simulation
from src.skfibers.experiments.survival_sim_simple_area import survival_data_area_simulation
from src.skfibers.experiments.survival_sim_3_groups import survival_data_simulation_3_groups
from src.skfibers.methods.bin import BIN
from src.skfibers.methods.util import (
    plot_optimal_bins_km_curves, 
    plot_bin_km_curve, 
    print_comparison_summary, 
    run_km_comparison, 
    find_optimal_bins_in_population,
    extend_curve_to_zero,
    calculate_area_between_curves,
    plot_curves_with_shaded_area,
    calculate_area_under_curve
)

current_working_directory = os.getcwd()
print(current_working_directory)

***
## Set Up Local Run Parameters


In [ ]:
local_save = False
folder_path = './output'
if not os.path.exists(folder_path):
        os.makedirs(folder_path)
if local_save:
    output_folder = './output'
else:
    output_folder = folder_path

***
## Generate Simulated Survial Data

In [ ]:
# generate data for 2 or 3 groups
dataset_num_groups = 3
# set ground-truth thresholds for 2 or 3 groups
two_thresh_list = [1,3]
one_thresh_list = [3]

In [ ]:
if dataset_num_groups == 2:
    data_name = '2-curve dataset'
    data = survival_data_simulation(instances=10000, total_features=100, predictive_features=10, low_risk_proportion=0.5, threshold=one_thresh_list[0], feature_frequency_range=(0.1, 0.4), 
                            noise_frequency=0.0, class0_time_to_event_range=(1.5, 0.1), class1_time_to_event_range=(1.25, 0.1), censoring_frequency=0.2, 
                            negative_control=False,random_seed=10)
    data.to_csv(output_folder+'/'+data_name+'.csv', index=False)
    data = pd.read_csv(output_folder+'/'+data_name+'.csv')
    true_risk_group = data[['TrueRiskGroup']]
    data = data.drop('TrueRiskGroup', axis=1)

    value_counts = true_risk_group['TrueRiskGroup'].value_counts()
    print(data)

In [ ]:
if dataset_num_groups == 3:
    data_name = '3-curve dataset'
    data = survival_data_simulation_3_groups(instances=10000, total_features=100, predictive_features=10, low_risk_proportion=0.5, med_risk_proportion=0.25, threshold1 = two_thresh_list[0], threshold2 = two_thresh_list[1], feature_frequency_range=(0.1, 0.4), 
                            noise_frequency=0.0, class0_time_to_event_range=(1.5, 0.1), class1_time_to_event_range=(1.25, 0.1), class2_time_to_event_range=(1, 0.1), censoring_frequency=0.2, 
                            covariates_to_sim=0, covariates_signal_range=(0.2,0.4), random_seed=10)
    data.to_csv(output_folder+'/'+data_name+'.csv', index=False)
    data = pd.read_csv(output_folder+'/'+data_name+'.csv')
    true_risk_group = data[['TrueRiskGroup']]
    data = data.drop('TrueRiskGroup', axis=1)

    value_counts = true_risk_group['TrueRiskGroup'].value_counts()
    print(data)

***
## Run FIBERS (Training)

## MULTI THRESHOLDING FIBERS

In [ ]:
try:
    fibers = FIBERS(outcome_label="Duration", outcome_type="survival", iterations=50, pop_size=50, tournament_prop=0.2, crossover_prob=0.5, 
                    min_mutation_prob=0.1, max_mutation_prob=0.5, merge_prob=0.1, new_gen=1.0, elitism=0.1, diversity_pressure=0, min_bin_size=1, 
                    max_bin_size=None, max_bin_init_size=10, fitness_metric="log_rank", log_rank_weighting=None, censor_label="Censoring", group_strata_min=0.4, 
                    penalty=0.5, group_thresh_list=None, min_thresh=0, max_thresh=5, int_thresh=True, thresh_evolve_prob=0.5, multi_thresholding=True,
                    manual_bin_init=None, covariates=None, pop_clean = 'group_strata', report=list(range(0, 50, 5)), random_seed=10,verbose=False)
    fibers = fibers.fit(data)
    print(f"Training completed. Model has {len(fibers.set.bin_pop)} bins")
except Exception as e:
    print(f"Training failed with error: {str(e)}")

***
## Statistics from Initial Training


In [ ]:
print(fibers.elapsed_time)

In [ ]:
bin_index=0

In [ ]:
# Get the single threshold for the best 2-group bin
# or we reach end of bin population
pos = 0
while len(fibers.set.bin_pop[pos].group_threshold_list) != 1 or pos == len(fibers.set.bin_pop) - 1:
    pos += 1
one_thresh_list = fibers.set.bin_pop[pos].group_threshold_list

if len(one_thresh_list) != 1:
    raise ValueError("No 2-group bin found in initial training")



In [ ]:
fibers.get_bin_report(bin_index)

In [ ]:
fibers.get_multi_kaplan_meir(data,bin_index,show=True,save=True,output_folder=output_folder,data_name=data_name)

In [ ]:
fibers.get_bin_population_heatmap_plot(save=True,output_folder=output_folder,data_name=data_name)

In [ ]:
fibers.get_threshold_progress_plot(save=True,output_folder=output_folder,data_name=data_name)

In [ ]:
fibers.get_misc_progress_plot(save=True,output_folder=output_folder,data_name=data_name)

In [ ]:
fibers.get_pareto_plot(save=True,output_folder=output_folder,data_name=data_name)

***
## Bin Injection


In [ ]:
# print top bins from 1st fibers run for comparison
print("Top 3 bins from initial training:")
for i in range(min(3, len(fibers.set.bin_pop))):
    bin_details = fibers.get_bin_report(i)
    print(f"Bin {i}:")
    print(bin_details)
    print("\n")

### Create Optimal Bin(s) and Evaluate

**Bin Injection Mechanism**: This section demonstrates how to create and evaluate optimal bins with predetermined thresholds. This is crucial for:

1. **Ground Truth Validation**: Testing if the algorithm can identify truly optimal bins
2. **Fitness Function Evaluation**: Comparing algorithm-discovered bins against known optimal solutions
3. **Performance Benchmarking**: Establishing baseline performance metrics

The `evaluate_fixed_bin()` method allows us to test specific bin configurations without going through the evolutionary process, enabling systematic evaluation of the multi-thresholding approach.

In [ ]:
# create bin that uses evaluate_fixed_bin method
optimal_bin = BIN(fibers.set.pareto)
optimal_bin.feature_list = ["P_" + str(i+1) for i in range(10)]
optimal_bin.group_threshold_list = two_thresh_list
optimal_bin.birth_iteration = 0
optimal_bin.bin_size = 10

# create bin that uses evaluate method
optimal_bin2 = BIN(fibers.set.pareto)
optimal_bin2.feature_list = ["P_" + str(i+1) for i in range(10)]
optimal_bin2.group_threshold_list = two_thresh_list
optimal_bin2.birth_iteration = 0
optimal_bin2.bin_size = 10

# print optimal bin report before evaluation - optimal_bin2 will have same report as optimal_bin
print("Optimal bin details (before evaluate):")
bin_report = optimal_bin.bin_report()
print(bin_report)
print("\n")

In [ ]:
# Evaluate the bin
optimal_bin.evaluate_fixed_bin(
    data.loc[:,fibers.feature_names], data.loc[:,fibers.outcome_label], data.loc[:,fibers.censor_label],
    fibers.outcome_type, fibers.fitness_metric, fibers.log_rank_weighting, fibers.outcome_label, fibers.censor_label, 
    fibers.min_thresh, fibers.max_thresh, fibers.int_thresh, fibers.group_thresh_list, False, fibers.multi_thresholding, 
    fibers.iterations, 0, fibers.residuals, data.loc[:, fibers.covariates if fibers.covariates else []],
    fibers.naive_survival_optimization, optimal_bin.group_threshold_list
)

optimal_bin2.evaluate(
    data.loc[:,fibers.feature_names], data.loc[:,fibers.outcome_label], data.loc[:,fibers.censor_label],
    fibers.outcome_type, fibers.fitness_metric, fibers.log_rank_weighting, fibers.outcome_label, fibers.censor_label, 
    fibers.min_thresh, fibers.max_thresh, fibers.int_thresh, fibers.group_thresh_list, False, fibers.multi_thresholding, 
    fibers.iterations, 0, fibers.residuals, data.loc[:, fibers.covariates if fibers.covariates else []],
    fibers.naive_survival_optimization
)

print("Optimal bin details (after evaluate_fixed_bin:")
bin_report = optimal_bin.bin_report()
print(bin_report)
print("\n")
print("Optimal bin2 details (after evaluate):")
bin_report2 = optimal_bin2.bin_report()
print(bin_report2)
print("\n")

In [ ]:
# Calculate pre-fitness
optimal_bin.calculate_pre_fitness(fibers.group_strata_min, fibers.penalty, fibers.fitness_metric, fibers.feature_names, fibers.naive_survival_optimization)
optimal_bin2.calculate_pre_fitness(fibers.group_strata_min, fibers.penalty, fibers.fitness_metric, fibers.feature_names, fibers.naive_survival_optimization)

print("Optimal bin details (after pre-fitness calculation):")
bin_report = optimal_bin.bin_report()
print(bin_report)
print("\n")
print("Optimal bin2 details (after pre-fitness calculation):")
bin_report2 = optimal_bin2.bin_report()
print(bin_report2)
print("\n")

### Inject Bins and Complete Fitness Update

In [ ]:
# Insert bin at the beginning of the bin population and update fitness
fibers.set.bin_pop.insert(0, optimal_bin)
fibers.set.global_fitness_update(fibers.penalty)

# Print top bin after fitness update
print("Top 3 bins after adding optimal bin and fitness update:")
for i in range(min(3, len(fibers.set.bin_pop))):
    bin_details = fibers.get_bin_report(i)
    print(f"Bin {i}:")
    print(bin_details)
    print("\n")

In [ ]:
# Create df for manual initialization
manual_bins_df = pd.DataFrame(columns=range(11))

# Add the optimal bin first at index 0
row_data = [str(optimal_bin.feature_list)]
row_data.append(optimal_bin.group_threshold_list[0])
for i in range(2, 11):
    if i == 10:
        row_data.append(0)
    else:
        row_data.append(None)
        
manual_bins_df.loc[0] = row_data

# Add the rest of the bins from the existing population
for i, bin_obj in enumerate(fibers.set.bin_pop[1:], 1):
    row_data = [str(bin_obj.feature_list)]
    
    if bin_obj.group_threshold_list and len(bin_obj.group_threshold_list) > 0:
        row_data.append(bin_obj.group_threshold_list[0])
    else:
        row_data.append(None)
        
    for j in range(2, 11):
        if j == 10:
            row_data.append(bin_obj.birth_iteration if hasattr(bin_obj, 'birth_iteration') else 0)
        else:
            row_data.append(None)
            
    manual_bins_df.loc[i] = row_data

# Save this to df
manual_bins_df.to_csv(output_folder+'/manual_bins.csv', index=False, header=False)
print(f"Saved manual bin population with {len(manual_bins_df)} bins")

proper_manual_bins = pd.read_csv(output_folder+'/manual_bins.csv', header=None)


### Second FIBERS Training

In [ ]:
num_iterations = 1

# new FIBERS instance
fibers_continued = FIBERS(outcome_label="Duration", outcome_type="survival", iterations=num_iterations, pop_size=50, tournament_prop=0.2, crossover_prob=0.5, 
                          min_mutation_prob=0.1, max_mutation_prob=0.5, merge_prob=0.1, new_gen=1.0, elitism=0.2, diversity_pressure=0, min_bin_size=1, 
                          max_bin_size=None, max_bin_init_size=10, fitness_metric="log_rank", log_rank_weighting=None, censor_label="Censoring", group_strata_min=0.4, 
                          penalty=0.5, group_thresh_list=None, min_thresh=0, max_thresh=5, int_thresh=True, thresh_evolve_prob=0.5, multi_thresholding=True, 
                          manual_bin_init=proper_manual_bins, covariates=None, pop_clean='group_strata', report=list(range(0, num_iterations)), random_seed=10, verbose=False)

fibers_continued = fibers_continued.fit(data)

# Print top bins from 2nd FIBERS run
print("Top 3 bins after continued training:")
for i in range(min(3, len(fibers_continued.set.bin_pop))):
    bin_details = fibers_continued.get_bin_report(i)
    print(f"Bin {i}:")
    print(bin_details)
    print("\n")

### Analyze New Bin Population

In [ ]:
# See how bins with all predictive features are doing
existing_optimal_bins = find_optimal_bins_in_population(fibers)

if existing_optimal_bins:
    print(f"{len(existing_optimal_bins)} bins with all predictive features:")
    for bin_info in existing_optimal_bins:
        print(f"  Index {bin_info['index']}: {bin_info['thresholds']}, "
              f"log-rank={bin_info['log_rank']:.1f}, fitness={bin_info['fitness']:.3f}")
        if bin_info['extra_features']:
            print(f"    Extra features: {bin_info['extra_features']}")
else:
    print("No bins with all predictive features found in current population")

In [ ]:
# Create and display KM curves for the optimal bins
print("Kaplan-Meier Curves for Optimal Bins:")
# need to set proper thresholds for the optimal bins in util.run_km_comparison
optimal_bin_3group, optimal_bin_2group = run_km_comparison(data, fibers, output_folder, data_name)


In [ ]:
# Comparing supposed optimal bins vs current top bins:
print(f"3-group optimal: {optimal_bin_3group.log_rank_score:.1f}")
print(f"2-group optimal: {optimal_bin_2group.log_rank_score:.1f}")
print(f"Current top bin: {fibers.set.bin_pop[0].log_rank_score:.1f}")

In [ ]:
# Create 3-group optimal bin
optimal_bin_3group = BIN(fibers.set.pareto)
optimal_bin_3group.feature_list = ["P_" + str(i+1) for i in range(10)]
optimal_bin_3group.group_threshold_list = two_thresh_list
optimal_bin_3group.birth_iteration = 0
optimal_bin_3group.bin_size = 10

# Create 2-group optimal bin
optimal_bin_2group = BIN(fibers.set.pareto)
optimal_bin_2group.feature_list = ["P_" + str(i+1) for i in range(10)]
optimal_bin_2group.group_threshold_list = one_thresh_list
optimal_bin_2group.birth_iteration = 0
optimal_bin_2group.bin_size = 10

# Evaluate both bins
for bin_obj, thresh_list, name in [(optimal_bin_3group, two_thresh_list, "3-group"), 
                                   (optimal_bin_2group, one_thresh_list, "2-group")]:
    bin_obj.evaluate_fixed_bin(
        data.loc[:,fibers.feature_names],
        data.loc[:,fibers.outcome_label],
        data.loc[:,fibers.censor_label],
        fibers.outcome_type,
        fibers.fitness_metric,
        fibers.log_rank_weighting,
        fibers.outcome_label,
        fibers.censor_label,
        fibers.min_thresh,
        fibers.max_thresh,
        fibers.int_thresh,
        fibers.group_thresh_list,
        False,
        fibers.multi_thresholding,
        fibers.iterations,
        0,
        fibers.residuals,
        data.loc[:, fibers.covariates if fibers.covariates else []],
        fibers.naive_survival_optimization,
        thresh_list
    )

# Calculate areas for 3 group bin and 2 group bin
print("\n3-Group Bin Areas:")
areas_3group, sf_3group = calculate_area_between_curves(data, optimal_bin_3group, fibers)

print("Areas between curves:")
for comparison, area in areas_3group.items():
    print(f"  {comparison}: {area:.4f}")

print("\nAreas under individual curves:")
for label, sf_data in sf_3group.items():
    area_under = calculate_area_under_curve(sf_data['times'], sf_data['probs'])
    print(f"  {label} risk curve: {area_under:.4f}")

print("\n2-Group Bin Areas:")
areas_2group, sf_2group = calculate_area_between_curves(data, optimal_bin_2group, fibers)

print("Areas between curves:")
for comparison, area in areas_2group.items():
    print(f"  {comparison}: {area:.4f}")

print("\nAreas under individual curves:")
for label, sf_data in sf_2group.items():
    area_under = calculate_area_under_curve(sf_data['times'], sf_data['probs'])
    print(f"  {label} risk curve: {area_under:.4f}")


# Plot KM Curves and areas between curves
area_3group_low_high = areas_3group['Low_vs_High']
area_2group_low_high = areas_2group['Low_vs_High']

areas_3group_plot, fig1 = plot_curves_with_shaded_area(
    data, optimal_bin_3group, fibers, 
    title_suffix=" - 3-Group Bin"
)
plt.savefig(f"{output_folder}/{data_name}_3group_area_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

areas_2group_plot, fig2 = plot_curves_with_shaded_area(
    data, optimal_bin_2group, fibers, 
    title_suffix=" - 2-Group Bin"
)
plt.savefig(f"{output_folder}/{data_name}_2group_area_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

***
## Statistics from Second Training

### Report Run Time

In [ ]:
print(fibers_continued.elapsed_time)

### Top (or Target) Bin Examination

In [ ]:
bin_index = 0 # lowest index is the bin with the highest fitness (only reports the bin ranked at the top, despite possible fitness ties for top)

### Get Bin Details

In [ ]:
fibers_continued.get_bin_report(0)

### Plot: Kaplan Meier Survival Curves For Each Group Defined by the Target Bin

In [ ]:
fibers_continued.get_multi_kaplan_meir(data,bin_index,show=True,save=True,output_folder=output_folder,data_name=data_name)

### Check and View Any Top Bin Ties

***
## Bin Population Examination
### Plot: Basic Bin Population Heatmap


In [ ]:
fibers_continued.get_bin_population_heatmap_plot(save=True,output_folder=output_folder,data_name=data_name)

### Plot: Custom Bin Population Heatmap

In [ ]:
"""
group_names=["P", "R"]
legend_group_info = ['Not in Bin','Non-Predictive Feature in Bin','Predictive Feature in Bin'] #2 default colors first followed by additional color descriptions in legend
color_features = [['P_1','P_2','P_3','P_4','P_5','P_6','P_7','P_8','P_9','P_10']]
colors = [(1, 0, 0)] # red ---Alternatively orange (1, 0.5, 0)
default_colors = [(.95, .95, 1),(0, 0, 1)] #very light blue and blue
max_bins = 100
max_features = 100

fibers.get_custom_bin_population_heatmap_plot(group_names,legend_group_info,color_features,colors,max_bins,max_features,save=True,output_folder=output_folder,data_name=data_name)
"""

### Plot: Bin Population Pareto Front
In plot, dot colors indicate the 'group strata prop' of the given bin, and dot size is relative to the 'group threshold of that bin'.

In [ ]:
fibers_continued.get_pareto_plot(save=True,output_folder=output_folder,data_name=data_name)

### Plot: Estimated Feature Tracking Scores
These scores accumulate throughout the training process, and do not nesessarily reflect feature importance of individual bins or the final bin population.

In [ ]:
fibers_continued.get_feature_tracking_plot(max_features=50,save=True,output_folder=output_folder,data_name=data_name)

### Plot: Dataset Covariate Residuals (if applicable)

In [ ]:
if fibers.fitness_metric == 'residuals' or fibers.fitness_metric == 'log_rank_residuals':  
    fibers.get_residuals_histogram(save=True,output_folder=output_folder,data_name=data_name)

### Plot: Bin Log-Rank Scores Vs. Residuals Scores (if applicable)
In plot, dot colors indicate the 'group strata prop' of the given bin, and dot size is relative to the 'group threshold of that bin'.

In [ ]:
if fibers.fitness_metric == 'log_rank_residuals':
    fibers.get_log_rank_residuals_plot(save=True,output_folder=output_folder,data_name=data_name)

###  Evaluate All Bins in Population using Cox PH Model (Can be slow)

In [ ]:
have_covariates = False
if have_covariates:
    fibers.calculate_cox_prop_hazards(data)

### Plot: Bin Log-Rank Scores Vs. Adjusted Hazard Ratios (if applicable)
In plot, dot colors indicate the 'group strata prop' of the given bin, and dot size is relative to the 'group threshold of that bin'.

In [ ]:
if have_covariates:
    if fibers.fitness_metric == 'log_rank' or fibers.fitness_metric == 'log_rank_residuals':  
        fibers.get_log_rank_adj_HR_plot(save=True,output_folder=output_folder,data_name=data_name)

### Plot: Bin Adjusted Hazard Ratios Vs. Residuals Scores (if applicable)
In plot, dot colors indicate the 'group strata prop' of the given bin, and dot size is relative to the 'group threshold of that bin'.

In [ ]:
if have_covariates:
    if fibers.fitness_metric == 'residuals' or fibers.fitness_metric == 'log_rank_residuals':   
        fibers.get_adj_HR_residuals_plot(save=True,output_folder=output_folder,data_name=data_name)

### Plot: Bin Adjusted Hazard Ratios Vs. Log Rank * Residuals Scores (if applicable)
In plot, dot colors indicate the 'group strata prop' of the given bin, and dot size is relative to the 'group threshold of that bin'.

In [ ]:
if have_covariates:
    if fibers.fitness_metric == 'log_rank_residuals':   
        fibers.get_adj_HR_metric_product_plot(save=True,output_folder=output_folder,data_name=data_name)

***
## History of Bin Evolution (Top Bin Each Generation)
### Plot: Pre-Fitness of top bin across training iterations

In [ ]:
fibers_continued.get_perform_progress_plot(save=True,output_folder=output_folder,data_name=data_name)

### Plot: Threshold of top bin across training iterations

In [ ]:
fibers_continued.get_threshold_progress_plot(save=True,output_folder=output_folder,data_name=data_name)

### Plot: Normalized Top-Bin Stats Across Training Iterations

In [ ]:
fibers_continued.get_misc_progress_plot(save=True,output_folder=output_folder,data_name=data_name)

### View Top Bin Information Across all Iterations/Generations

In [ ]:
fibers.perform_track_df
fibers.perform_track_df.to_csv(output_folder+'/'+'Tracking_'+data_name+'.csv', index=False)

***
## Save Bin Population
### Save Bin Population Details to CSV

In [ ]:
pop_df = fibers.get_pop()
pop_df.to_csv(output_folder+'/'+'Pop_'+data_name+'.csv', index=False)

### Pickle Trained FIBERS Object (For Future Use)

In [ ]:
model_name = "pareto"
with open(output_folder+'/'+model_name+'.pickle', 'wb') as f:
    pickle.dump(fibers_pareto, f)

***
## Transforming Bins Into New Features (i.e. Feature Learning) and Save as New CSV Files
### Transform Bins Using Total Sums (i.e. Respective Bin Thresholds Not Applied)

In [ ]:
tdf = fibers.transform(data,full_sums=True)
tdf.to_csv(output_folder+'/'+'Transformed_FullSums_'+data_name+'.csv', index=False)
tdf

### Transform Bins Using Respective Bin Threshold (i.e. 0 = At/Under Threshold Group and 1 = Over Threshold Group)

In [ ]:
tdf = fibers.transform(data,full_sums=False)
tdf.to_csv(output_folder+'/'+'Transformed_Threshold'+data_name+'.csv', index=False)
tdf

***
## Prediction (of Group/Strata)
### Predict Strata (Low vs. High) Using Top Bin

In [ ]:
predictions = fibers.predict(data,bin_number=0)
print(classification_report(predictions, true_risk_group, digits=8))


### Predict Strata (Low vs. High) Using Whole Bin Population (Weighted Voting Scheme)
Assuming that a single bin can best solve the target survival problem (as is the case in this simulation), we expect prediction by all bins with this weighted voting scheme to perform less well.

In [ ]:
predictions = fibers.predict(data)
print(classification_report(predictions, true_risk_group, digits=8))

***
## Open Pickled FIBERS Object (Example)

In [ ]:
with open(output_folder+'/'+data_name+'.pickle', 'rb') as f:
    fibers = pickle.load(f)

fibers.get_bin_report(bin_index)